# Visualize an MITgcm Held-Suarez run

Reads the Zarr written by `datagen.mitgcm.held_suarez.scripts.run` and renders
snapshots, Hovmöller diagrams, and animations of the 500 hPa state.

Dataset layout: `(time, field, lat, lon)` with
`field ∈ {u_500hpa, v_500hpa, T_500hpa, ps}`. Time is in seconds since
the start of the data-collection phase (spin-up is not included).

## Environment

Needs `xarray`, `zarr`, `numpy`, `matplotlib`, `cartopy`, `imageio[ffmpeg]`.
All ship with the `datagen` project env.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

import os

DATA_PATH = Path(os.environ.get(
    "DATA_ROOT",
    f"{os.environ.get('SCRATCH', '/tmp')}/flowers-data",
)) / "mitgcm" / "short_run" / "run.zarr"

ds = xr.open_zarr(DATA_PATH)
ds

In [ ]:
print("Dims:", dict(ds.sizes))
print("Time range:", float(ds.time.min()) / 86400, "..",
      float(ds.time.max()) / 86400, "days")
print("Lat range: ", float(ds.lat.min()), "..", float(ds.lat.max()), "deg")
print("Lon range: ", float(ds.lon.min()), "..", float(ds.lon.max()), "deg")
print("Fields:    ", [str(f) for f in ds.field.values])
print("Parameters:")
for k, v in ds.attrs.items():
    if k.startswith("param_"):
        print(f"  {k[6:]:14s} = {v}")

## Snapshot of all four fields

First saved snapshot (data-collection start, after the spin-up). Even
after a few model days the 500 hPa flow shows realistic mid-latitude
westerlies and developing baroclinic eddies; the surface pressure has
small zonal-mean structure since the Held-Suarez problem conserves
global mass exactly.

In [ ]:
lat = ds.lat.values
lon = ds.lon.values

def plot_field(ax, arr, title, cmap="RdBu_r", symmetric=True, units=""):
    if symmetric:
        vmax = float(np.nanmax(np.abs(arr)))
        vmin = -vmax
    else:
        vmin, vmax = float(np.nanmin(arr)), float(np.nanmax(arr))
    im = ax.pcolormesh(lon, lat, arr, cmap=cmap, vmin=vmin, vmax=vmax,
                       shading="auto")
    ax.set_title(title)
    ax.set_xlabel("lon [deg]")
    ax.set_ylabel("lat [deg]")
    plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02, label=units)

t0 = 0
fig, axes = plt.subplots(2, 2, figsize=(14, 7), constrained_layout=True)
plot_field(axes[0, 0],
           ds.fields.sel(field="u_500hpa").isel(time=t0).values,
           "u at 500 hPa (zonal)", units="m/s")
plot_field(axes[0, 1],
           ds.fields.sel(field="v_500hpa").isel(time=t0).values,
           "v at 500 hPa (meridional)", units="m/s")
plot_field(axes[1, 0],
           ds.fields.sel(field="T_500hpa").isel(time=t0).values,
           "θ at 500 hPa", cmap="viridis", symmetric=False, units="K")
plot_field(axes[1, 1],
           ds.fields.sel(field="ps").isel(time=t0).values / 100.0,
           "surface pressure", cmap="viridis", symmetric=False, units="hPa")
fig.suptitle(f"t = {float(ds.time.isel(time=t0)) / 86400:.2f} days")
plt.show()

## Time evolution of u at 500 hPa

Six evenly-spaced panels showing how the upper-tropospheric zonal wind
develops eddy structure on top of the underlying westerly jets in both
hemispheres.

In [ ]:
u_all = ds.fields.sel(field="u_500hpa").values
n_times = ds.sizes["time"]
panels = np.linspace(0, n_times - 1, 6, dtype=int)
vmax_u = float(np.nanmax(np.abs(u_all)))

fig, axes = plt.subplots(2, 3, figsize=(18, 8),
                         constrained_layout=True, sharex=True, sharey=True)
for ax, ti in zip(axes.flat, panels):
    im = ax.pcolormesh(lon, lat, u_all[ti], cmap="RdBu_r",
                       vmin=-vmax_u, vmax=vmax_u, shading="auto")
    ax.set_title(f"t = {float(ds.time.isel(time=ti)) / 86400:.2f} d")
    ax.set_xlabel("lon [deg]")
    ax.set_ylabel("lat [deg]")
fig.colorbar(im, ax=axes, fraction=0.015, pad=0.02, label="u [m/s]")
fig.suptitle("u at 500 hPa")
plt.show()

## Zonal-mean Hovmöller diagrams

Zonal mean over longitude as a function of `(time, lat)`. The Held-Suarez
benchmark is statistically zonally symmetric, so the long-time mean of
the zonal wind should show two westerly jets at ~30–45° in each hemisphere
and weak easterlies in the deep tropics. The zonal-mean θ should display
a smooth equator-to-pole gradient close to the prescribed `delta_T_y`.

In [ ]:
u_zm = ds.fields.sel(field="u_500hpa").mean("lon")
T_zm = ds.fields.sel(field="T_500hpa").mean("lon")
t_d = ds.time.values / 86400.0

fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
vmax = float(np.abs(u_zm).max())
im0 = axes[0].pcolormesh(t_d, lat, u_zm.T, cmap="RdBu_r",
                          vmin=-vmax, vmax=vmax, shading="auto")
axes[0].set_title("Zonal-mean u at 500 hPa [m/s]")
axes[0].set_xlabel("time [days]")
axes[0].set_ylabel("lat [deg]")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].pcolormesh(t_d, lat, T_zm.T, cmap="viridis", shading="auto")
axes[1].set_title("Zonal-mean θ at 500 hPa [K]")
axes[1].set_xlabel("time [days]")
axes[1].set_ylabel("lat [deg]")
fig.colorbar(im1, ax=axes[1])
plt.show()

## Globally-integrated diagnostics

Area-weighted RMS of u and global max of |u|, plus area-weighted mean
surface pressure (must stay essentially constant — MITgcm conserves dry
atmospheric mass under `exactConserv=.TRUE.`).

In [ ]:
lat_rad = np.deg2rad(lat)
w = np.cos(lat_rad)
w = w / w.sum()

u = ds.fields.sel(field="u_500hpa").values
v = ds.fields.sel(field="v_500hpa").values
ps = ds.fields.sel(field="ps").values
speed = np.sqrt(u ** 2 + v ** 2)

rms_u = np.sqrt((u ** 2).mean(axis=-1) @ w)
max_speed = speed.max(axis=(-1, -2))
ps_mean = (ps.mean(axis=-1) @ w) / 100.0

fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
axes[0].plot(t_d, rms_u)
axes[0].set_xlabel("time [days]")
axes[0].set_ylabel("RMS u [m/s]")
axes[0].set_title("Area-weighted RMS u at 500 hPa")
axes[1].plot(t_d, max_speed)
axes[1].set_xlabel("time [days]")
axes[1].set_ylabel("max |u| [m/s]")
axes[1].set_title("Global max wind speed at 500 hPa")
axes[2].plot(t_d, ps_mean)
axes[2].set_xlabel("time [days]")
axes[2].set_ylabel("<p_s> [hPa]")
axes[2].set_title("Area-weighted mean surface pressure")
plt.show()

## u animation (flat MP4)

In [ ]:
import imageio.v3 as iio
from tqdm import trange

fps = 10
dpi = 100
output_path = Path("mitgcm_u500.mp4")

frames = []
for i in trange(u_all.shape[0]):
    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True, dpi=dpi)
    mesh = ax.pcolormesh(lon, lat, u_all[i], cmap="RdBu_r",
                         vmin=-vmax_u, vmax=vmax_u, shading="auto")
    ax.set_title(f"u at 500 hPa | t = {float(ds.time.isel(time=i)) / 86400:.2f} d")
    ax.set_xlabel("lon [deg]")
    ax.set_ylabel("lat [deg]")
    fig.colorbar(mesh, ax=ax, label="u [m/s]")

    fig.canvas.draw()
    w_px, h_px = fig.canvas.get_width_height()
    frame = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h_px, w_px, 4)
    frame = frame[: h_px - (h_px % 2), : w_px - (w_px % 2), :3]
    frames.append(frame)
    plt.close(fig)

iio.imwrite(
    str(output_path),
    np.stack(frames, axis=0),
    fps=fps,
    codec="libx264",
    pixelformat="yuv420p",
)
print(f"Wrote {output_path}  ({len(frames)} frames @ {fps} fps)")

## u animation (spherical, rotating globe)

Cartopy orthographic projection with a slow rotation around the central
longitude over the run.

In [ ]:
import cartopy.crs as ccrs
from tqdm import trange

fps = 10
dpi = 100
output_path = Path("mitgcm_u500_spherical.mp4")

num_frames = u_all.shape[0]
frames = []

for i in trange(num_frames):
    fig = plt.figure(figsize=(8, 8), constrained_layout=True, dpi=dpi)
    cent_lon = (i / num_frames) * 360.0 - 180.0
    ax = fig.add_subplot(
        1, 1, 1,
        projection=ccrs.Orthographic(central_longitude=cent_lon, central_latitude=20.0),
    )
    ax.set_global()
    ax.gridlines(alpha=0.3, linestyle="--")

    mesh = ax.pcolormesh(
        lon, lat, u_all[i],
        cmap="RdBu_r",
        vmin=-vmax_u, vmax=vmax_u,
        shading="auto",
        transform=ccrs.PlateCarree(),
    )
    ax.set_title(f"MITgcm u@500 hPa | t = {float(ds.time.isel(time=i)) / 86400:.2f} d")
    fig.colorbar(mesh, ax=ax, shrink=0.6, label="u [m/s]", pad=0.05)

    fig.canvas.draw()
    w_px, h_px = fig.canvas.get_width_height()
    frame = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h_px, w_px, 4)
    frame = frame[: h_px - (h_px % 2), : w_px - (w_px % 2), :3]
    frames.append(frame)
    plt.close(fig)

iio.imwrite(
    str(output_path),
    np.stack(frames, axis=0),
    fps=fps,
    codec="libx264",
    pixelformat="yuv420p",
)
print(f"Wrote {output_path}  ({len(frames)} frames @ {fps} fps)")